# 🛒 E-Commerce Data Cleaning & Machine Learning Pipeline

Pipeline ini memproses dataset `e_commerce.csv`, menangani missing values, melakukan segmentasi (K-Means), dan analisis penyebab pembatalan (Random Forest).

**Fitur utama:**
- Menampilkan tabel data **Before** dan **After** cleaning secara realtime
- Perbandingan statistik sebelum dan sesudah pembersihan data

In [1]:
import pandas as pd
import numpy as np
import json
import os
from IPython.display import display, HTML, Markdown
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer

# Path konfigurasi
input_csv = "../e_commerce.csv"
output_dir = "output"
os.makedirs(output_dir, exist_ok=True)
output_csv = os.path.join(output_dir, "cleaned_ecommerce.csv")
output_rf_json = os.path.join(output_dir, "rf_feature_importance.json")
output_report_json = os.path.join(output_dir, "cleaning_report.json")

# Helper function untuk styling tabel
def styled_header(text, color='#3b82f6'):
    display(HTML(f'<h3 style="color:{color}; border-bottom: 2px solid {color}; padding-bottom: 8px; margin-top: 20px;">{text}</h3>'))

def highlight_missing(val):
    """Highlight missing values dengan warna merah"""
    if pd.isna(val) or val == '' or val is None:
        return 'background-color: #fecaca; color: #991b1b; font-weight: bold'
    return ''

print(f"Loading data from {input_csv}...")
df_raw = pd.read_csv(input_csv, delimiter=';')
print(f"✅ Data loaded successfully!")
print(f"📊 Shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")

Loading data from ../e_commerce.csv...
✅ Data loaded successfully!
📊 Shape: 20848 rows × 19 columns


---
## 📋 BEFORE CLEANING — Data Awal (Raw Data)
Menampilkan kondisi data sebelum proses cleaning dilakukan.

In [2]:
# === BEFORE CLEANING: Preview Data Awal ===
styled_header('📄 Preview 10 Baris Pertama — Data Mentah (BEFORE)', '#ef4444')

# Tampilkan tabel dengan highlight missing values
display(
    df_raw.head(10)
    .style
    .map(highlight_missing)
    .set_caption('Data Mentah — Sel merah = Missing Value')
    .set_table_styles([
        {'selector': 'caption', 'props': [('font-size', '14px'), ('font-weight', 'bold'), ('color', '#dc2626')]},
        {'selector': 'th', 'props': [('background-color', '#1e293b'), ('color', 'white'), ('font-size', '12px'), ('padding', '8px')]},
        {'selector': 'td', 'props': [('font-size', '11px'), ('padding', '6px 8px')]},
    ])
)

,order_id,total_qty,total_weight_gr,total_returned_qty,Total Diskon,product_categories,num_product_categories,Status Pesanan,Alasan Pembatalan,Opsi Pengiriman,Metode Pembayaran,Kota/Kabupaten,Provinsi,Ongkos Kirim Dibayar oleh Pembeli,Estimasi Potongan Biaya Pengiriman,Total Pembayaran,Perkiraan Ongkos Kirim,Waktu Pesanan Dibuat,source_file
0,ORD_0000001,2,2000,0,0,Celengan,1,Selesai,nan,Reguler (Cashless)-SPX Standard,Saldo ShopeePay,KOTA SERANG,BANTEN,0,10000,38300,10000,2024-04-01 00:15,AprilSales2024.xlsx
1,ORD_0000002,1,500,0,0,Celengan,1,Selesai,nan,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA SEMARANG,JAWA TENGAH,0,14500,18576,14500,2024-04-01 01:47,AprilSales2024.xlsx
2,ORD_0000003,1,500,0,0,Celengan,1,Selesai,nan,Hemat Kargo-SPX Hemat,SeaBank Bayar Instan,KAB. BOGOR,JAWA BARAT,0,8000,7069,8000,2024-04-01 04:25,AprilSales2024.xlsx
3,ORD_0000004,2,400,0,0,Mangkok Sambal / Saus,1,Selesai,nan,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA JAMBI,JAMBI,0,20000,32200,20000,2024-04-01 04:41,AprilSales2024.xlsx
4,ORD_0000005,3,3600,0,0,"Keranjang, Other, Tempat Nasi",3,Batal,Dibatalkan oleh Pembeli. Alasan: Ubah Pesanan yang Ada,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA TANGERANG,BANTEN,0,0,0,8000,2024-04-01 06:12,AprilSales2024.xlsx
5,ORD_0000006,2,1000,0,0,Celengan,1,Selesai,nan,Hemat Kargo-SPX Hemat,SPayLater,KAB. BANDUNG,JAWA BARAT,0,10000,40996,10000,2024-04-01 07:09,AprilSales2024.xlsx
6,ORD_0000007,1,1700,0,0,Nampan / Tray,1,Batal,Dibatalkan oleh Pembeli. Alasan: Ubah Pesanan yang Ada,Hemat Kargo-SPX Hemat,SPayLater,KOTA TANGERANG,BANTEN,0,0,0,16000,2024-04-01 07:23,AprilSales2024.xlsx
7,ORD_0000008,1,500,0,0,Celengan,1,Selesai,nan,Hemat Kargo-SPX Hemat,Online Payment,KAB. BOGOR,JAWA BARAT,0,8000,18280,8000,2024-04-01 07:33,AprilSales2024.xlsx
8,ORD_0000009,1,500,0,0,Celengan,1,Batal,Dibatalkan oleh Pembeli. Alasan: Ubah Pesanan yang Ada,Hemat Kargo-J&T Economy,Online Payment,KAB. GARUT,JAWA BARAT,0,0,0,16500,2024-04-01 07:56,AprilSales2024.xlsx
9,ORD_0000010,6,12900,0,0,"Aksesoris Mandi, Celengan, Keranjang",3,Selesai,nan,Hemat Kargo-SPX Hemat,Online Payment,KAB. TEGAL,JAWA TENGAH,13000,20000,92800,33000,2024-04-01 08:27,AprilSales2024.xlsx


In [3]:
# === BEFORE CLEANING: Info Data ===
styled_header('📊 Informasi Data — BEFORE Cleaning', '#ef4444')

# Buat tabel info missing values
before_info = pd.DataFrame({
    'Kolom': df_raw.columns,
    'Tipe Data': df_raw.dtypes.values,
    'Non-Null Count': df_raw.notna().sum().values,
    'Missing Count': df_raw.isna().sum().values,
    'Missing %': (df_raw.isna().sum().values / len(df_raw) * 100).round(2)
})

# Highlight kolom yang punya missing values
def highlight_missing_rows(row):
    if row['Missing Count'] > 0:
        return ['background-color: #fef2f2'] * len(row)
    return [''] * len(row)

display(
    before_info
    .style
    .apply(highlight_missing_rows, axis=1)
    .set_caption(f'Info Data Awal — Total: {df_raw.shape[0]} baris, {df_raw.shape[1]} kolom')
    .set_table_styles([
        {'selector': 'caption', 'props': [('font-size', '14px'), ('font-weight', 'bold')]},
        {'selector': 'th', 'props': [('background-color', '#1e293b'), ('color', 'white'), ('padding', '8px')]},
        {'selector': 'td', 'props': [('padding', '6px 8px')]},
    ])
    .hide(axis='index')
)

# Ringkasan missing values
total_missing_before = df_raw.isna().sum().sum()
cols_with_missing = df_raw.columns[df_raw.isna().any()].tolist()

print(f"\n🔴 Total Missing Values: {total_missing_before}")
print(f"🔴 Kolom dengan Missing: {cols_with_missing if cols_with_missing else 'Tidak ada'}")

Kolom,Tipe Data,Non-Null Count,Missing Count,Missing %
order_id,str,20848,0,0.000000
total_qty,int64,20848,0,0.000000
total_weight_gr,int64,20848,0,0.000000
total_returned_qty,int64,20848,0,0.000000
Total Diskon,int64,20848,0,0.000000
product_categories,str,20848,0,0.000000
num_product_categories,int64,20848,0,0.000000
Status Pesanan,str,20848,0,0.000000
Alasan Pembatalan,str,2830,18018,86.430000
Opsi Pengiriman,str,20848,0,0.000000



🔴 Total Missing Values: 19998
🔴 Kolom dengan Missing: ['Alasan Pembatalan', 'Waktu Pesanan Dibuat']


In [4]:
# === BEFORE CLEANING: Statistik Deskriptif ===
styled_header('📈 Statistik Deskriptif — BEFORE Cleaning', '#ef4444')

before_stats = df_raw.describe(include='all').T
display(
    before_stats
    .style
    .set_caption('Statistik Deskriptif Data Mentah')
    .set_table_styles([
        {'selector': 'caption', 'props': [('font-size', '14px'), ('font-weight', 'bold')]},
        {'selector': 'th', 'props': [('background-color', '#1e293b'), ('color', 'white'), ('padding', '8px')]},
        {'selector': 'td', 'props': [('padding', '6px 8px'), ('font-size', '11px')]},
    ])
)

# Simpan bentuk awal untuk perbandingan
shape_before = df_raw.shape
missing_before_detail = df_raw.isna().sum()
missing_before_detail = missing_before_detail[missing_before_detail > 0]
duplicates_before = df_raw.duplicated().sum()

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
order_id,20848,20848,ORD_0000001,1,nan,nan,nan,nan,nan,nan,nan
total_qty,20848.000000,nan,nan,nan,2.560821,7.796763,1.000000,1.000000,1.000000,2.000000,256.000000
total_weight_gr,20848.000000,nan,nan,nan,2004.529691,7106.357515,10.000000,300.000000,500.000000,1600.000000,375000.000000
total_returned_qty,20848.000000,nan,nan,nan,0.017748,0.548656,0.000000,0.000000,0.000000,0.000000,70.000000
Total Diskon,20848.000000,nan,nan,nan,405.199348,9784.017705,0.000000,0.000000,0.000000,0.000000,700000.000000
product_categories,20848,679,Celengan,5514,nan,nan,nan,nan,nan,nan,nan
num_product_categories,20848.000000,nan,nan,nan,1.112097,0.485361,1.000000,1.000000,1.000000,1.000000,11.000000
Status Pesanan,20848,11,Selesai,17768,nan,nan,nan,nan,nan,nan,nan
Alasan Pembatalan,2830,18,Dibatalkan oleh Pembeli. Alasan: Lainnya/ berubah pikiran,662,nan,nan,nan,nan,nan,nan,nan
Opsi Pengiriman,20848,45,Hemat Kargo-SPX Hemat,12597,nan,nan,nan,nan,nan,nan,nan


---
## 🧹 1. Data Cleaning Process
Proses pembersihan data meliputi:
1. Handling Missing Values
2. Konversi tipe data
3. Pengecekan duplikat
4. Validasi konsistensi data

In [5]:
# Buat copy untuk dikerjakan
df = df_raw.copy()

styled_header('🔧 Proses Cleaning Dimulai...', '#f59e0b')

# --- Step 1: Menangani Missing Values ---
print("\n📌 Step 1: Menangani Missing Values")
print("="*50)

# 1. Alasan Pembatalan: Isi dengan 'Tidak Batal'
missing_alasan = df['Alasan Pembatalan'].isna().sum()
df['Alasan Pembatalan'] = df['Alasan Pembatalan'].fillna('Tidak Batal')
print(f"  ✅ 'Alasan Pembatalan': {missing_alasan} NaN → diisi 'Tidak Batal'")

# 2. Waktu Pesanan Dibuat: Konversi datetime
missing_waktu_before = df['Waktu Pesanan Dibuat'].isna().sum()
df['Waktu Pesanan Dibuat'] = pd.to_datetime(df['Waktu Pesanan Dibuat'], errors='coerce')
missing_waktu_coerced = df['Waktu Pesanan Dibuat'].isna().sum()

# Drop baris yang datetime-nya NaT karena krusial untuk time-series dashboard
rows_before_drop = len(df)
df = df.dropna(subset=['Waktu Pesanan Dibuat'])
rows_dropped = rows_before_drop - len(df)
print(f"  ✅ 'Waktu Pesanan Dibuat': {missing_waktu_before} NaN awal, {missing_waktu_coerced} setelah coerce")
print(f"     → {rows_dropped} baris di-drop karena datetime tidak valid")

# --- Step 2: Konversi Kolom Numerik ---
print("\n📌 Step 2: Konversi Kolom Numerik")
print("="*50)

numeric_cols = ['total_qty', 'total_weight_gr', 'total_returned_qty', 'Total Diskon', 
                'num_product_categories', 'Ongkos Kirim Dibayar oleh Pembeli', 
                'Estimasi Potongan Biaya Pengiriman', 'Total Pembayaran', 'Perkiraan Ongkos Kirim']

for col in numeric_cols:
    non_numeric_count = pd.to_numeric(df[col], errors='coerce').isna().sum() - df[col].isna().sum()
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    if non_numeric_count > 0:
        print(f"  ✅ '{col}': {non_numeric_count} nilai non-numerik dikonversi → 0")
    else:
        print(f"  ✅ '{col}': OK (sudah numerik)")

# --- Step 3: Duplikat ---
print("\n📌 Step 3: Pengecekan Duplikat")
print("="*50)
dupes = df.duplicated().sum()
if dupes > 0:
    df = df.drop_duplicates()
    print(f"  ✅ {dupes} baris duplikat dihapus")
else:
    print(f"  ✅ Tidak ditemukan baris duplikat")

print(f"\n✅ Cleaning selesai! Shape: {df.shape[0]} rows × {df.shape[1]} columns")


📌 Step 1: Menangani Missing Values
  ✅ 'Alasan Pembatalan': 18018 NaN → diisi 'Tidak Batal'
  ✅ 'Waktu Pesanan Dibuat': 1980 NaN awal, 1980 setelah coerce
     → 1980 baris di-drop karena datetime tidak valid

📌 Step 2: Konversi Kolom Numerik
  ✅ 'total_qty': OK (sudah numerik)
  ✅ 'total_weight_gr': OK (sudah numerik)
  ✅ 'total_returned_qty': OK (sudah numerik)
  ✅ 'Total Diskon': OK (sudah numerik)
  ✅ 'num_product_categories': OK (sudah numerik)
  ✅ 'Ongkos Kirim Dibayar oleh Pembeli': OK (sudah numerik)
  ✅ 'Estimasi Potongan Biaya Pengiriman': OK (sudah numerik)
  ✅ 'Total Pembayaran': OK (sudah numerik)
  ✅ 'Perkiraan Ongkos Kirim': OK (sudah numerik)

📌 Step 3: Pengecekan Duplikat
  ✅ Tidak ditemukan baris duplikat

✅ Cleaning selesai! Shape: 18868 rows × 19 columns


---
## ✅ AFTER CLEANING — Data Setelah Dibersihkan
Menampilkan kondisi data setelah seluruh proses cleaning dilakukan.

In [6]:
# === AFTER CLEANING: Preview Data Bersih ===
styled_header('📄 Preview 10 Baris Pertama — Data Bersih (AFTER)', '#10b981')

display(
    df.head(10)
    .style
    .map(highlight_missing)
    .set_caption('Data Setelah Cleaning — Tidak ada sel merah = Bersih!')
    .set_table_styles([
        {'selector': 'caption', 'props': [('font-size', '14px'), ('font-weight', 'bold'), ('color', '#059669')]},
        {'selector': 'th', 'props': [('background-color', '#064e3b'), ('color', 'white'), ('font-size', '12px'), ('padding', '8px')]},
        {'selector': 'td', 'props': [('font-size', '11px'), ('padding', '6px 8px')]},
    ])
)

,order_id,total_qty,total_weight_gr,total_returned_qty,Total Diskon,product_categories,num_product_categories,Status Pesanan,Alasan Pembatalan,Opsi Pengiriman,Metode Pembayaran,Kota/Kabupaten,Provinsi,Ongkos Kirim Dibayar oleh Pembeli,Estimasi Potongan Biaya Pengiriman,Total Pembayaran,Perkiraan Ongkos Kirim,Waktu Pesanan Dibuat,source_file
0,ORD_0000001,2,2000,0,0,Celengan,1,Selesai,Tidak Batal,Reguler (Cashless)-SPX Standard,Saldo ShopeePay,KOTA SERANG,BANTEN,0,10000,38300,10000,2024-04-01 00:15:00,AprilSales2024.xlsx
1,ORD_0000002,1,500,0,0,Celengan,1,Selesai,Tidak Batal,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA SEMARANG,JAWA TENGAH,0,14500,18576,14500,2024-04-01 01:47:00,AprilSales2024.xlsx
2,ORD_0000003,1,500,0,0,Celengan,1,Selesai,Tidak Batal,Hemat Kargo-SPX Hemat,SeaBank Bayar Instan,KAB. BOGOR,JAWA BARAT,0,8000,7069,8000,2024-04-01 04:25:00,AprilSales2024.xlsx
3,ORD_0000004,2,400,0,0,Mangkok Sambal / Saus,1,Selesai,Tidak Batal,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA JAMBI,JAMBI,0,20000,32200,20000,2024-04-01 04:41:00,AprilSales2024.xlsx
4,ORD_0000005,3,3600,0,0,"Keranjang, Other, Tempat Nasi",3,Batal,Dibatalkan oleh Pembeli. Alasan: Ubah Pesanan yang Ada,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA TANGERANG,BANTEN,0,0,0,8000,2024-04-01 06:12:00,AprilSales2024.xlsx
5,ORD_0000006,2,1000,0,0,Celengan,1,Selesai,Tidak Batal,Hemat Kargo-SPX Hemat,SPayLater,KAB. BANDUNG,JAWA BARAT,0,10000,40996,10000,2024-04-01 07:09:00,AprilSales2024.xlsx
6,ORD_0000007,1,1700,0,0,Nampan / Tray,1,Batal,Dibatalkan oleh Pembeli. Alasan: Ubah Pesanan yang Ada,Hemat Kargo-SPX Hemat,SPayLater,KOTA TANGERANG,BANTEN,0,0,0,16000,2024-04-01 07:23:00,AprilSales2024.xlsx
7,ORD_0000008,1,500,0,0,Celengan,1,Selesai,Tidak Batal,Hemat Kargo-SPX Hemat,Online Payment,KAB. BOGOR,JAWA BARAT,0,8000,18280,8000,2024-04-01 07:33:00,AprilSales2024.xlsx
8,ORD_0000009,1,500,0,0,Celengan,1,Batal,Dibatalkan oleh Pembeli. Alasan: Ubah Pesanan yang Ada,Hemat Kargo-J&T Economy,Online Payment,KAB. GARUT,JAWA BARAT,0,0,0,16500,2024-04-01 07:56:00,AprilSales2024.xlsx
9,ORD_0000010,6,12900,0,0,"Aksesoris Mandi, Celengan, Keranjang",3,Selesai,Tidak Batal,Hemat Kargo-SPX Hemat,Online Payment,KAB. TEGAL,JAWA TENGAH,13000,20000,92800,33000,2024-04-01 08:27:00,AprilSales2024.xlsx


In [7]:
# === AFTER CLEANING: Info Data ===
styled_header('📊 Informasi Data — AFTER Cleaning', '#10b981')

after_info = pd.DataFrame({
    'Kolom': df.columns,
    'Tipe Data': df.dtypes.values,
    'Non-Null Count': df.notna().sum().values,
    'Missing Count': df.isna().sum().values,
    'Missing %': (df.isna().sum().values / len(df) * 100).round(2)
})

display(
    after_info
    .style
    .set_caption(f'Info Data Bersih — Total: {df.shape[0]} baris, {df.shape[1]} kolom')
    .set_table_styles([
        {'selector': 'caption', 'props': [('font-size', '14px'), ('font-weight', 'bold'), ('color', '#059669')]},
        {'selector': 'th', 'props': [('background-color', '#064e3b'), ('color', 'white'), ('padding', '8px')]},
        {'selector': 'td', 'props': [('padding', '6px 8px')]},
    ])
    .hide(axis='index')
)

total_missing_after = df.isna().sum().sum()
print(f"\n🟢 Total Missing Values: {total_missing_after}")
print(f"🟢 Shape: {df.shape[0]} baris × {df.shape[1]} kolom")

Kolom,Tipe Data,Non-Null Count,Missing Count,Missing %
order_id,str,18868,0,0.000000
total_qty,int64,18868,0,0.000000
total_weight_gr,int64,18868,0,0.000000
total_returned_qty,int64,18868,0,0.000000
Total Diskon,int64,18868,0,0.000000
product_categories,str,18868,0,0.000000
num_product_categories,int64,18868,0,0.000000
Status Pesanan,str,18868,0,0.000000
Alasan Pembatalan,str,18868,0,0.000000
Opsi Pengiriman,str,18868,0,0.000000



🟢 Total Missing Values: 0
🟢 Shape: 18868 baris × 19 kolom


In [8]:
# === AFTER CLEANING: Statistik Deskriptif ===
styled_header('📈 Statistik Deskriptif — AFTER Cleaning', '#10b981')

after_stats = df.describe(include='all').T
display(
    after_stats
    .style
    .set_caption('Statistik Deskriptif Data Bersih')
    .set_table_styles([
        {'selector': 'caption', 'props': [('font-size', '14px'), ('font-weight', 'bold'), ('color', '#059669')]},
        {'selector': 'th', 'props': [('background-color', '#064e3b'), ('color', 'white'), ('padding', '8px')]},
        {'selector': 'td', 'props': [('padding', '6px 8px'), ('font-size', '11px')]},
    ])
)

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
order_id,18868,18868,ORD_0000001,1,nan,nan,nan,nan,nan,nan,nan
total_qty,18868.000000,nan,nan,nan,2.572186,1.000000,1.000000,1.000000,2.000000,256.000000,7.843661
total_weight_gr,18868.000000,nan,nan,nan,2021.273055,10.000000,300.000000,500.000000,1600.000000,375000.000000,7044.934507
total_returned_qty,18868.000000,nan,nan,nan,0.018709,0.000000,0.000000,0.000000,0.000000,70.000000,0.575668
Total Diskon,18868.000000,nan,nan,nan,424.220691,0.000000,0.000000,0.000000,0.000000,700000.000000,10110.492750
product_categories,18868,640,Celengan,5083,nan,nan,nan,nan,nan,nan,nan
num_product_categories,18868.000000,nan,nan,nan,1.113473,1.000000,1.000000,1.000000,1.000000,11.000000,0.491409
Status Pesanan,18868,11,Selesai,16045,nan,nan,nan,nan,nan,nan,nan
Alasan Pembatalan,18868,19,Tidak Batal,16295,nan,nan,nan,nan,nan,nan,nan
Opsi Pengiriman,18868,45,Hemat Kargo-SPX Hemat,11571,nan,nan,nan,nan,nan,nan,nan


---
## 🔄 Perbandingan BEFORE vs AFTER Cleaning
Tabel ringkasan perbedaan data sebelum dan sesudah proses cleaning.

In [9]:
# === TABEL PERBANDINGAN BEFORE vs AFTER ===
styled_header('🔄 Ringkasan Perbandingan: Before vs After', '#3b82f6')

shape_after = df.shape
missing_after_total = df.isna().sum().sum()
duplicates_after = df.duplicated().sum()

comparison_data = {
    'Metrik': [
        'Jumlah Baris',
        'Jumlah Kolom', 
        'Total Missing Values',
        'Jumlah Duplikat',
        'Baris Dihapus',
        'Missing Values Diperbaiki',
    ],
    'BEFORE 🔴': [
        f"{shape_before[0]:,}",
        f"{shape_before[1]}",
        f"{total_missing_before:,}",
        f"{duplicates_before:,}",
        '—',
        '—',
    ],
    'AFTER 🟢': [
        f"{shape_after[0]:,}",
        f"{shape_after[1]}",
        f"{missing_after_total:,}",
        f"{duplicates_after:,}",
        f"{shape_before[0] - shape_after[0]:,}",
        f"{total_missing_before - missing_after_total:,}",
    ],
    'Status': [
        '✅' if shape_after[0] > 0 else '❌',
        '✅',
        '✅ Bersih!' if missing_after_total == 0 else f'⚠️ Masih ada {missing_after_total}',
        '✅ Bersih!' if duplicates_after == 0 else f'⚠️ Masih ada {duplicates_after}',
        '✅',
        '✅',
    ]
}

comparison_df = pd.DataFrame(comparison_data)

display(
    comparison_df
    .style
    .set_caption('Perbandingan Data: Before vs After Cleaning')
    .set_table_styles([
        {'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold'), ('color', '#1e40af'), ('padding', '12px')]},
        {'selector': 'th', 'props': [('background-color', '#1e3a5f'), ('color', 'white'), ('padding', '10px 14px'), ('font-size', '13px')]},
        {'selector': 'td', 'props': [('padding', '10px 14px'), ('font-size', '13px'), ('border-bottom', '1px solid #e2e8f0')]},
    ])
    .hide(axis='index')
)

# Tabel detail per-kolom Missing Values
if len(missing_before_detail) > 0:
    styled_header('📋 Detail Missing Values Per Kolom', '#3b82f6')
    
    missing_comparison = pd.DataFrame({
        'Kolom': missing_before_detail.index,
        'Missing BEFORE': missing_before_detail.values,
        '% BEFORE': (missing_before_detail.values / shape_before[0] * 100).round(2),
        'Missing AFTER': [df[col].isna().sum() if col in df.columns else 'N/A' for col in missing_before_detail.index],
        'Penanganan': [
            'Diisi "Tidak Batal"' if col == 'Alasan Pembatalan' 
            else 'Drop baris NaT' if col == 'Waktu Pesanan Dibuat'
            else 'to_numeric + fillna(0)' if col in numeric_cols
            else 'Otomatis'
            for col in missing_before_detail.index
        ]
    })
    
    display(
        missing_comparison
        .style
        .set_caption('Detail Penanganan Missing Values')
        .set_table_styles([
            {'selector': 'caption', 'props': [('font-size', '14px'), ('font-weight', 'bold')]},
            {'selector': 'th', 'props': [('background-color', '#1e3a5f'), ('color', 'white'), ('padding', '8px')]},
            {'selector': 'td', 'props': [('padding', '8px')]},
        ])
        .hide(axis='index')
    )
else:
    print("✅ Tidak ada missing values pada data awal.")

print("\n" + "="*60)
print("📊 KESIMPULAN CLEANING:")
print(f"   • Data awal: {shape_before[0]:,} baris → Data bersih: {shape_after[0]:,} baris")
print(f"   • Missing values: {total_missing_before:,} → {missing_after_total:,}")
print(f"   • Duplikat: {duplicates_before:,} → {duplicates_after:,}")
print("="*60)

Metrik,BEFORE 🔴,AFTER 🟢,Status
Jumlah Baris,"20,848","18,868",✅
Jumlah Kolom,19,19,✅
Total Missing Values,"19,998",0,✅ Bersih!
Jumlah Duplikat,0,0,✅ Bersih!
Baris Dihapus,—,"1,980",✅
Missing Values Diperbaiki,—,"19,998",✅


Kolom,Missing BEFORE,% BEFORE,Missing AFTER,Penanganan
Alasan Pembatalan,18018,86.430000,0,"Diisi ""Tidak Batal"""
Waktu Pesanan Dibuat,1980,9.500000,0,Drop baris NaT



📊 KESIMPULAN CLEANING:
   • Data awal: 20,848 baris → Data bersih: 18,868 baris
   • Missing values: 19,998 → 0
   • Duplikat: 0 → 0


---
## 2. K-Means Clustering (Segmentasi Pelanggan/Pesanan)

In [10]:
# Kita akan mensegmentasi pesanan berdasarkan perilaku pembelian
cluster_features = ['total_qty', 'total_weight_gr', 'Total Pembayaran', 'Perkiraan Ongkos Kirim']
X_cluster = df[cluster_features].copy()

# Standarisasi data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# K-Means dengan k=4 (misal: budget, standard, bulk, premium)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

# Penamaan cluster berdasar rata-rata Total Pembayaran
cluster_means = df.groupby('Cluster')['Total Pembayaran'].mean().sort_values()
cluster_mapping = {
    cluster_means.index[0]: 'Budget Order',
    cluster_means.index[1]: 'Standard Order',
    cluster_means.index[2]: 'High Value Order',
    cluster_means.index[3]: 'Bulk/Premium Order'
}
df['Cluster_Label'] = df['Cluster'].map(cluster_mapping)

styled_header('🎯 Distribusi Cluster', '#8b5cf6')
cluster_dist = df['Cluster_Label'].value_counts().reset_index()
cluster_dist.columns = ['Cluster', 'Jumlah Pesanan']
cluster_dist['Persentase'] = (cluster_dist['Jumlah Pesanan'] / len(df) * 100).round(2).astype(str) + '%'

display(
    cluster_dist
    .style
    .set_caption('Segmentasi Pesanan (K-Means, k=4)')
    .set_table_styles([
        {'selector': 'caption', 'props': [('font-size', '14px'), ('font-weight', 'bold')]},
        {'selector': 'th', 'props': [('background-color', '#4c1d95'), ('color', 'white'), ('padding', '10px')]},
        {'selector': 'td', 'props': [('padding', '8px')]},
    ])
    .hide(axis='index')
)

Cluster,Jumlah Pesanan,Persentase
Budget Order,17186,91.09%
Standard Order,1513,8.02%
Bulk/Premium Order,114,0.6%
High Value Order,55,0.29%


---
## 3. Random Forest (Analisis Pembatalan)

In [11]:
# Membuat model untuk memprediksi apakah pesanan Batal
# Target: Batal (1) vs Tidak Batal (0)
df['Is_Batal'] = (df['Status Pesanan'] != 'Selesai').astype(int)

# Fitur yang digunakan
rf_features = ['total_qty', 'total_weight_gr', 'Total Pembayaran', 'Metode Pembayaran', 'Opsi Pengiriman', 'Provinsi']

# Siapkan data untuk Random Forest
X_rf = df[rf_features].copy()
y_rf = df['Is_Batal']

# Label Encoding untuk kolom kategorikal
le_dict = {}
cat_features = ['Metode Pembayaran', 'Opsi Pengiriman', 'Provinsi']
for col in cat_features:
    le = LabelEncoder()
    # Handle missing value di kategorikal (just in case)
    X_rf[col] = X_rf[col].fillna('Unknown')
    X_rf[col] = le.fit_transform(X_rf[col].astype(str))
    le_dict[col] = le

# Train Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_rf, y_rf)

# Ambil Feature Importance
importances = rf.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': rf_features,
    'Importance': importances
}).sort_values('Importance', ascending=False)

styled_header('🌲 Feature Importance (Random Forest)', '#059669')

fi_display = feature_importance_df.copy()
fi_display['Importance'] = fi_display['Importance'].round(4)
fi_display['Bar'] = fi_display['Importance'].apply(
    lambda x: '█' * int(x * 50) + '░' * (50 - int(x * 50))
)

display(
    fi_display
    .style
    .set_caption('Faktor Penyebab Pembatalan Pesanan')
    .set_table_styles([
        {'selector': 'caption', 'props': [('font-size', '14px'), ('font-weight', 'bold')]},
        {'selector': 'th', 'props': [('background-color', '#064e3b'), ('color', 'white'), ('padding', '10px')]},
        {'selector': 'td', 'props': [('padding', '8px'), ('font-family', 'monospace')]},
    ])
    .hide(axis='index')
)

# Simpan ke JSON untuk Dashboard
rf_result = {
    'features': feature_importance_df['Feature'].tolist(),
    'importances': feature_importance_df['Importance'].tolist()
}
with open(output_rf_json, 'w') as f:
    json.dump(rf_result, f)
print(f"\n✅ Feature importance saved to {output_rf_json}")

Feature,Importance,Bar
Total Pembayaran,0.887900,████████████████████████████████████████████░░░░░░
Opsi Pengiriman,0.072000,███░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
Provinsi,0.013600,░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
total_weight_gr,0.011400,░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
Metode Pembayaran,0.009800,░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
total_qty,0.005400,░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░



✅ Feature importance saved to output\rf_feature_importance.json


---
## 4. Export Data & Cleaning Report

In [12]:
# Export data yang sudah dibersihkan dan dilabeli cluster ke CSV
# Untuk Dashboard Laravel
# Format tanggal diseragamkan
df['Waktu Pesanan Dibuat'] = df['Waktu Pesanan Dibuat'].dt.strftime('%Y-%m-%d %H:%M:%S')

df.to_csv(output_csv, index=False)
print(f"✅ Cleaned dataset saved to {output_csv}")

# === Generate Cleaning Report JSON untuk Dashboard Laravel ===

# === COMPUTE ADDITIONAL METRICS FOR DASHBOARD ===
import numpy as np
import pandas as pd

# 1. Outlier Detection (IQR)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
outlier_details = {}
total_outlier_rows = set()

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower) | (df[col] > upper)].index
    outlier_count = len(outliers)
    if outlier_count > 0:
        total_outlier_rows.update(outliers)
        outlier_details[col] = {
            'Q1': float(Q1),
            'Q3': float(Q3),
            'IQR': float(IQR),
            'lower_bound': float(lower),
            'upper_bound': float(upper),
            'outlier_count': int(outlier_count),
            'outlier_percentage': round(outlier_count / len(df) * 100, 2)
        }

outliers_data = {
    'method': 'IQR',
    'total_outlier_rows': len(total_outlier_rows),
    'total_clean_rows': len(df) - len(total_outlier_rows),
    'per_column': outlier_details
}

# 2. Consistency
consistency_data = {
    'total_fixes': 0,
    'categories_normalized': True,
    'out_of_range': {}
}

# 3. Statistics
stats_numeric = df[numeric_cols].describe().to_dict()
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
stats_categorical = {col: df[col].value_counts().head(5).to_dict() for col in categorical_cols}

statistics_data = {
    'total_rows': int(len(df)),
    'numeric': {
        col: {
            'mean': round(stats['mean'], 2) if not pd.isna(stats['mean']) else 0,
            'median': round(stats['50%'], 2) if not pd.isna(stats['50%']) else 0,
            'std': round(stats['std'], 2) if not pd.isna(stats['std']) else 0,
            'min': round(stats['min'], 2) if not pd.isna(stats['min']) else 0,
            'max': round(stats['max'], 2) if not pd.isna(stats['max']) else 0,
            'q1': round(stats['25%'], 2) if not pd.isna(stats['25%']) else 0,
            'q3': round(stats['75%'], 2) if not pd.isna(stats['75%']) else 0,
        } for col, stats in stats_numeric.items()
    },
    'categorical': stats_categorical
}

cleaning_report = {
    'source_file': 'e_commerce.csv',
    'original_shape': {
        'rows': int(shape_before[0]),
        'columns': int(shape_before[1])
    },
    'missing_values': {
        'before': {
            'total': int(total_missing_before),
            'per_column': {col: int(val) for col, val in missing_before_detail.items()}
        },
        'after': {
            'total': int(missing_after_total),
            'per_column': {col: int(df[col].isna().sum()) for col in df.columns if df[col].isna().sum() > 0}
        },
        'resolved': int(total_missing_before - missing_after_total),
        'method': 'fillna (kategorikal), dropna (datetime), to_numeric + fillna(0) (numerik)'
    },
    'duplicates': {
        'before': int(shape_before[0]),
        'after': int(df.shape[0]),
        'removed': int(duplicates_before),
        'example_ids': []
    },
    'final_shape': {
        'rows': int(df.shape[0]),
        'columns': int(df.shape[1])
    },
    'sample_before': df_raw.head(5).fillna('NaN').to_dict(orient='records'),
        'sample_after': df.head(5).to_dict(orient='records'),
    'outliers': outliers_data,
    'consistency': consistency_data,
    'statistics': statistics_data
}

with open(output_report_json, 'w') as f:
    json.dump(cleaning_report, f, indent=2, default=str)
print(f"✅ Cleaning report saved to {output_report_json}")

print("\n" + "="*60)
print("🎉 Pipeline selesai!")
print(f"   📁 Data bersih: {output_csv}")
print(f"   📁 Feature importance: {output_rf_json}")
print(f"   📁 Cleaning report: {output_report_json}")
print("="*60)

✅ Cleaned dataset saved to output\cleaned_ecommerce.csv
✅ Cleaning report saved to output\cleaning_report.json

🎉 Pipeline selesai!
   📁 Data bersih: output\cleaned_ecommerce.csv
   📁 Feature importance: output\rf_feature_importance.json
   📁 Cleaning report: output\cleaning_report.json


C:\Users\Satrio Z\AppData\Local\Temp\ipykernel_5088\636247447.py:57: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
